In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model="Gemma2-9b-It",
    api_key=groq_api_key
)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001DAF9A6F0B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DAF9BBAAE0>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage
response = model.invoke(
    [
        HumanMessage(content="Hi, my name is Murat. What is your name?"),
        HumanMessage(content="What is the capital of Turkey?")
    ]
)
response
print(response.content)

The capital of Turkey is **Ankara**.  



In [5]:
from langchain_core.messages import AIMessage
response = model.invoke(
    [HumanMessage(
        content="What is the capital of Turkey?",
    ),AIMessage(
        content="The capital of Turkey is Ankara."
    ),
    HumanMessage(
        content="Hi, my name is Murat"
    )]
)

In [6]:
# Chat History
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history
)

In [7]:
config = {"configurable": {"session_id": "chat_session_1"}}


In [8]:
response = with_message_history.invoke(
    [HumanMessage(
        content="What is the capital of Turkey?"
    )],
    config=config
)
response.content

'The capital of Turkey is **Ankara**. \n'

In [9]:
with_message_history.invoke(
    [HumanMessage(
        content="Ankara capital of where?"
    )],
    config=config
)

AIMessage(content='Ankara is the capital of **Turkey**. \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 41, 'total_tokens': 53, 'completion_time': 0.021818182, 'prompt_time': 0.00255263, 'queue_time': 0.160440226, 'total_time': 0.024370812}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--61d6155c-a61b-4d13-bf2f-a001da4910be-0', usage_metadata={'input_tokens': 41, 'output_tokens': 12, 'total_tokens': 53})

In [10]:
# change the config to a different session
config1 = {"configurable": {"session_id": "chat_session_2"}}
response = with_message_history.invoke(
    [HumanMessage(
        content="What is my name?"
    )],
    config=config1
)
response.content

"As a large language model, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to know!\n"

In [11]:
response = with_message_history.invoke(
    [HumanMessage(
        content="Hey, my name is Murat"
    )],
    config=config1
)
response.content

"Hi Murat! It's nice to meet you. How can I help you today?\n"

In [12]:
response = with_message_history.invoke(
    [HumanMessage(
        content="What is my name?"
    )],
    config=config1
)
response.content

'Your name is Murat.  \n\nI remember that you told me earlier! 😊  Is there anything else I can help you with?\n'

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="messages")
    ]
)   

chain = prompt | model

In [14]:
chain.invoke(
    {"messages": [
        HumanMessage(content="Hi, my name is Murat. What is your name?"),
        HumanMessage(content="What is the capital of Turkey?")
    ]}  )

AIMessage(content='The capital of Turkey is Ankara.  \n\nIs there anything else I can help you with, Murat?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 44, 'total_tokens': 68, 'completion_time': 0.043636364, 'prompt_time': 0.003536149, 'queue_time': 0.16220637000000002, 'total_time': 0.047172513}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--1d1e6530-f165-47d5-88f4-354c0c56c56c-0', usage_metadata={'input_tokens': 44, 'output_tokens': 24, 'total_tokens': 68})

In [15]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history
)


In [16]:
config = {"configurable": {"session_id": "chat_session_1"}}
response = with_message_history.invoke(
    [HumanMessage(
        content="hi there, my name is Murat."
    )],
    config=config
)
response.content

"Hello Murat, it's nice to meet you! 👋  \n\nHow can I help you today?\n"

In [17]:
# Add more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all questions of your best your ability in {language}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt | model

In [18]:
response = chain.invoke(
    {"messages": [
        HumanMessage(content="Hi, my name is Murat."),
        HumanMessage(content="What is the capital of Turkey?"),
        HumanMessage(content="Please answer in Turkish.")
    ],"language": "Turkish"}
)
print(response.content)

Türkiye'nin başkenti Ankara'dır. 



In [19]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)




In [24]:
config = {"configurable": {"session_id": "chat4"}}
response = with_message_history.invoke(
    {'messages': [HumanMessage(
        content="hi there, my name is Murat."
    )],"language": "Turkish"},
    config=config
)
response.content

'Merhaba Murat! Benim adım Bard, sana yardımcı olmak için buradayım. Sorularına elimden geldiğince cevaplamaya çalışacağım. 😊 \n\nNe hakkında konuşmak istersin?\n\n'

In [25]:
response = with_message_history.invoke(
    {'messages': [HumanMessage(
        content="what is my name?"
    )],"language": "Turkish"},
    config=config
)
response.content

'Adınız Murat. 😉 \n\nBen bunu ilk mesajınızdan hatırlıyorum! \n\n\n'

In [31]:
# Manage conversations history

from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the capital of Turkey?"),
    AIMessage(content="The capital of Turkey is Ankara."),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of France is Paris."),
    HumanMessage(content="What is my name?"),
    AIMessage(content="Your name is Murat.")
]

trimmer.invoke(messages)

[HumanMessage(content='What is the capital of Turkey?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The capital of Turkey is Ankara.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your name is Murat.', additional_kwargs={}, response_metadata={})]

In [40]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain =  (RunnablePassthrough.assign(messages = itemgetter("messages")) |
          prompt |
          model )

response = chain.invoke(
    {"messages": [
        HumanMessage(content="Hi, my name is Murat"),
        HumanMessage(content="What is the capital of Turkey?")
    ],"language": "Turkish"}
)
response.content

"Türkiye'nin başkenti **Ankara**'dır. \n"

In [33]:
# Wrap this in the message history
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable": {"session_id": "chat5"}}

In [39]:
response = with_message_history.invoke(
    {'messages': [HumanMessage(content="What is my name?"),
                  ],
     "language": "Turkish"},
    config=config
)   

response.content

'Üzgünüm, ben senin adını bilmiyorum.  İsimimi sormaktan çekinme! \n'